# Summa Mathematica — Colab OCR pipeline (PDF → raw Markdown)

Part 1 of the pipeline. **OCR only**: `marker-pdf` converts the PDFs to Markdown, and the result is pushed to
`raw_ocr/` of the **private** repository `math_books` with the commit message `chore: add raw OCR markdown`.
No LLM / DeepSeek API is called here (marker's `--use_llm` is not used).

Before the first run:
1. Runtime → Change runtime type → **T4 GPU** (CPU works but is very slow).
2. Colab secrets (🔑 in the left bar) → add `GITHUB_TOKEN`: a fine-grained personal access token with
   *Contents: Read and write* on `sirisaacrewton-glitch/math_books` only. Enable “Notebook access”.
3. Colab secrets → add `DRIVE_FOLDER_ID`: the ID of the Google Drive folder holding the PDFs
   (the part after `/folders/` in its URL). It is kept out of this notebook because the notebook is public.

Book text must never be pushed to the public repository `summa-mathematica`; this notebook only writes to `math_books`.

In [ ]:
# ===== Settings =====
from google.colab import userdata
DRIVE_FOLDER_ID  = userdata.get("DRIVE_FOLDER_ID").strip()   # Colab secret (not written here: this file is public)
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/" + DRIVE_FOLDER_ID

# How to get the PDFs:
#   "api"   : sign in with google.colab.auth and download the folder by its ID with the Drive API
#             (works for private folders and shared folders; recommended. Does not mount the drive)
#   "mount" : mount your Google Drive and find the folder by its ID (works for private folders;
#             the folder must be in My Drive or added to it as a shortcut)
#   "gdown" : download the folder by URL (the folder must be shared as "anyone with the link"; max 50 files)
SOURCE_MODE = "api"      # "mount" falls back to "api" if mounting fails ("credential propagation was unsuccessful")
# mount only, optional: the folder path after mounting, e.g. "/content/drive/MyDrive/math_pdfs".
# Leave empty to find the folder by DRIVE_FOLDER_ID; set it if that search fails or is slow.
DRIVE_FOLDER_PATH = ""

GITHUB_REPO   = "sirisaacrewton-glitch/math_books"   # PRIVATE repository (book text allowed)
GITHUB_BRANCH = "main"
OUTPUT_DIR    = "raw_ocr"                              # inside the repository
COMMIT_MESSAGE = "chore: add raw OCR markdown"

ONLY_FILES    = []      # e.g. ["klingenberg-diffgeom.pdf"]; empty = every PDF in the folder
PAGE_RANGE    = None    # e.g. "0-40" (0-based, marker syntax) or None for all pages
FORCE_OCR     = False   # True: OCR every page even if the PDF has a text layer (scanned books)
SKIP_EXISTING = True    # skip PDFs whose raw_ocr/<name>.md already exists in the repository

In [ ]:
# ===== Install (takes a few minutes; restart is not needed) =====
!pip -q install "marker-pdf>=1.8" gdown
import torch, importlib.metadata as md
print("marker-pdf", md.version("marker-pdf"), "| CUDA:", torch.cuda.is_available())

In [ ]:
# ===== Collect the PDFs =====
import os, glob, pathlib, shutil, subprocess

PDF_DIR = "/content/pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

def find_folder_by_id(root, folder_id):
    # Drive exposes the file ID as an extended attribute on mounted folders.
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            p = os.path.join(dirpath, d)
            try:
                if os.getxattr(p, "user.drive.id").decode() == folder_id:
                    return p
            except OSError:
                pass
    return None

def download_via_api(folder_id, dest):
    # Drive API (v3) download of every PDF in the folder and its subfolders.
    import logging
    # harmless notice from the auth library ("httplib2 transport does not support per-request timeout")
    logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    service = build("drive", "v3")
    def children(fid):
        token = None
        while True:
            r = service.files().list(q="'%s' in parents and trashed = false" % fid,
                                     fields="nextPageToken, files(id, name, mimeType, shortcutDetails)",
                                     pageSize=1000, pageToken=token,
                                     supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            yield from r.get("files", [])
            token = r.get("nextPageToken")
            if not token:
                return
    stack, n = [folder_id], 0
    while stack:
        for f in children(stack.pop()):
            fid, mime = f["id"], f["mimeType"]
            if mime == "application/vnd.google-apps.shortcut":
                fid, mime = f["shortcutDetails"]["targetId"], f["shortcutDetails"]["targetMimeType"]
            if mime == "application/vnd.google-apps.folder":
                stack.append(fid)
                continue
            if not f["name"].lower().endswith(".pdf"):
                continue
            if ONLY_FILES and f["name"] not in ONLY_FILES:
                continue
            out = os.path.join(dest, f["name"])
            with open(out, "wb") as fh:
                dl = MediaIoBaseDownload(fh, service.files().get_media(fileId=fid, supportsAllDrives=True),
                                         chunksize=50 * 1024 * 1024)
                done = False
                while not done:
                    _, done = dl.next_chunk()
            n += 1
            print("downloaded", f["name"])
    print(n, "file(s) downloaded")

if SOURCE_MODE == "mount":
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:  # e.g. MessageError: credential propagation was unsuccessful
        print("drive.mount failed (%s); falling back to SOURCE_MODE = 'api'" % e)
        SOURCE_MODE = "api"

if SOURCE_MODE == "api":
    download_via_api(DRIVE_FOLDER_ID, PDF_DIR)
elif SOURCE_MODE == "mount":
    src = DRIVE_FOLDER_PATH if DRIVE_FOLDER_PATH and os.path.isdir(DRIVE_FOLDER_PATH) else \
          find_folder_by_id("/content/drive/MyDrive", DRIVE_FOLDER_ID) or \
          find_folder_by_id("/content/drive/Shareddrives", DRIVE_FOLDER_ID)
    if src is None:
        raise SystemExit("Folder not found in My Drive. Add a shortcut to it in My Drive "
                         "(right click → Organize → Add shortcut), set DRIVE_FOLDER_PATH, "
                         "or use SOURCE_MODE = 'gdown'.")
    print("PDF folder:", src)
    for p in glob.glob(os.path.join(src, "**", "*.pdf"), recursive=True):
        shutil.copy2(p, PDF_DIR)
else:
    import gdown
    gdown.download_folder(DRIVE_FOLDER_URL, output=PDF_DIR, quiet=False, use_cookies=False)

pdfs = sorted(glob.glob(os.path.join(PDF_DIR, "**", "*.pdf"), recursive=True))
if ONLY_FILES:
    pdfs = [p for p in pdfs if os.path.basename(p) in ONLY_FILES]
print(len(pdfs), "PDF(s):", [os.path.basename(p) for p in pdfs])

In [ ]:
# ===== Clone the private repository =====
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_DIR = "/content/repo"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

def run(cmd, **kw):
    # never echo the token
    r = subprocess.run(cmd, cwd=kw.get("cwd"), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(("command failed: %s\n%s" % (" ".join(cmd), r.stderr)).replace(TOKEN, "***"))
    return r.stdout

remote = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, GITHUB_REPO)
run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, remote, REPO_DIR])
run(["git", "config", "user.name", "colab-ocr"], cwd=REPO_DIR)
run(["git", "config", "user.email", "colab-ocr@users.noreply.github.com"], cwd=REPO_DIR)
os.makedirs(os.path.join(REPO_DIR, OUTPUT_DIR), exist_ok=True)
print("cloned", GITHUB_REPO, GITHUB_BRANCH)

In [ ]:
# ===== OCR with marker-pdf (no LLM) =====
import datetime, re, time
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

options = {"output_format": "markdown", "disable_image_extraction": True, "force_ocr": FORCE_OCR}
if PAGE_RANGE:
    options["page_range"] = PAGE_RANGE
config = ConfigParser(options)
converter = PdfConverter(
    artifact_dict=create_model_dict(),
    config=config.generate_config_dict(),
    processor_list=config.get_processors(),
    renderer=config.get_renderer(),
    llm_service=None,           # OCR only: no LLM calls
)

def out_name(pdf):
    stem = pathlib.Path(pdf).stem
    return re.sub(r"[^A-Za-z0-9._-]+", "-", stem).strip("-") + ".md"

def push(paths):
    run(["git", "add", *paths], cwd=REPO_DIR)
    if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR).returncode == 0:
        return
    run(["git", "commit", "-m", COMMIT_MESSAGE], cwd=REPO_DIR)
    for i in range(4):
        try:
            run(["git", "pull", "--rebase", "origin", GITHUB_BRANCH], cwd=REPO_DIR)
            run(["git", "push", "origin", "HEAD:" + GITHUB_BRANCH], cwd=REPO_DIR)
            return
        except RuntimeError as e:
            print("push failed, retrying:", str(e)[:200])
            time.sleep(2 ** (i + 1))
    raise RuntimeError("push failed 4 times")

for pdf in pdfs:
    rel = os.path.join(OUTPUT_DIR, out_name(pdf))
    dst = os.path.join(REPO_DIR, rel)
    if SKIP_EXISTING and os.path.exists(dst):
        print("skip (exists):", rel)
        continue
    t0 = time.time()
    rendered = converter(pdf)
    text, _, _ = text_from_rendered(rendered)
    header = "<!-- raw OCR by marker-pdf %s from %s on %s (pages: %s, force_ocr: %s). Unedited. -->\n\n" % (
        md.version("marker-pdf"), os.path.basename(pdf), datetime.date.today().isoformat(),
        PAGE_RANGE or "all", FORCE_OCR)
    with open(dst, "w", encoding="utf-8") as f:
        f.write(header + text)
    push([rel])   # one commit per book, so a disconnect does not lose finished work
    print("pushed %s (%d chars, %.0f s)" % (rel, len(text), time.time() - t0))
print("done")